In [1]:
%pip install transformers
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Local Inference on GPU
Model page: https://huggingface.co/facebook/sam3

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/sam3)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [2]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.11.0+cu130
CUDA available: True


In [3]:
from huggingface_hub import login
# login()

In [4]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("mask-generation", model="facebook/sam3")

Loading weights:   0%|          | 0/685 [00:00<?, ?it/s]

In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("facebook/sam3")
model = AutoModel.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

In [6]:
### segment all images in all timestep folders inside one parent folder ###
### skips images that already have a saved mask ###

# !pip install -q transformers torch pillow numpy

from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import numpy as np
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# ======================
# SETTINGS
# ======================

parent_dir = Path("content/consecutive_timesteps_10")

text_prompt = "plant"
threshold = 0.5
mask_threshold = 0.5
crop_padding = 160

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# ======================
# HELPERS
# ======================

def run_sam_text(image):
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=threshold,
        mask_threshold=mask_threshold,
        target_sizes=inputs["original_sizes"].tolist()
    )[0]

    return results


def select_best_result(results):
    if len(results["scores"]) == 0:
        return None

    if isinstance(results["scores"], torch.Tensor):
        idx = int(torch.argmax(results["scores"]).item())
    else:
        idx = int(np.argmax(results["scores"]))

    return {
        "box": results["boxes"][idx],
        "mask": results["masks"][idx],
    }


def to_numpy_mask(mask):
    if isinstance(mask, torch.Tensor):
        return mask.cpu().numpy().astype(np.uint8)
    return np.array(mask).astype(np.uint8)


def to_numpy_box(box):
    if isinstance(box, torch.Tensor):
        return box.cpu().numpy().astype(int)
    return np.array(box).astype(int)


def expand_box(box, W, H, pad):
    x1, y1, x2, y2 = box
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(W, x2 + pad)
    y2 = min(H, y2 + pad)
    return [x1, y1, x2, y2]


# ======================
# PROCESS ONE IMAGE
# ======================

def process_image(image_path, output_dir):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"Failed: {image_path} ({e})")
        return False

    W, H = image.size

    # First pass
    results = run_sam_text(image)
    best = select_best_result(results)

    if best is None:
        print("No detection:", image_path.name)
        return False

    box = to_numpy_box(best["box"])
    mask_full = to_numpy_mask(best["mask"])

    # Crop refinement
    x1, y1, x2, y2 = expand_box(box, W, H, crop_padding)
    crop = image.crop((x1, y1, x2, y2))

    results_crop = run_sam_text(crop)
    best_crop = select_best_result(results_crop)

    if best_crop is None:
        final_mask = mask_full
    else:
        crop_mask = to_numpy_mask(best_crop["mask"])
        final_mask = np.zeros((H, W), dtype=np.uint8)
        final_mask[y1:y2, x1:x2] = crop_mask

    mask_img = Image.fromarray((final_mask > 0).astype(np.uint8) * 255)

    output_path = output_dir / f"{image_path.stem}_mask_sam3.png"
    mask_img.save(output_path)

    print("Saved:", output_path)
    return True


# ======================
# FIND TIMESTEP FOLDERS
# ======================

timestep_dirs = sorted([
    p for p in parent_dir.iterdir()
    if p.is_dir()
])

print(f"Found {len(timestep_dirs)} timestep folders")

# ======================
# RUN ALL FOLDERS
# ======================

total_processed = 0
total_skipped = 0

for timestep_dir in timestep_dirs:
    input_dir = timestep_dir / "images"
    output_dir = timestep_dir / "masks"

    if not input_dir.exists():
        print(f"Skipping {timestep_dir.name}: no images folder")
        continue

    output_dir.mkdir(parents=True, exist_ok=True)

    images = [
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    ]

    print(f"\nProcessing folder: {timestep_dir.name}")
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Found {len(images)} images")

    folder_processed = 0
    folder_skipped = 0

    for img in images:
        output_path = output_dir / f"{img.stem}_mask_sam3.png"

        if output_path.exists():
            print(f"Skipping (already done): {img.name}")
            folder_skipped += 1
            total_skipped += 1
            continue

        success = process_image(img, output_dir)
        if success:
            folder_processed += 1
            total_processed += 1

    print(
        f"Finished {timestep_dir.name}: "
        f"{folder_processed} new, {folder_skipped} skipped"
    )

print(f"\nFinished all folders: {total_processed} new, {total_skipped} skipped.")

Using device: cuda


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

Found 11 timestep folders
Skipping .ipynb_checkpoints: no images folder

Processing folder: timestep_2004
Input:  content/consecutive_timesteps_10/timestep_2004/images
Output: content/consecutive_timesteps_10/timestep_2004/masks
Found 22 images
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010094_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010153_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010035_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010083_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010173_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010014_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_timesteps_10/timestep_2004/masks/GX010132_20250912_120302_2006_mask_sam3.png
Saved: content/consecutive_